

1. 01_only_text.csv
2. 02_only_speech.csv
3. 03_text_speech.csv
4. 04_only_eeg.csv
5. 05_text_speech_eeg.csv


- usa GuideOfTimes.csv para enlazar cada carpeta de conversación con su avatar
- añade label desde phq_label_por_usuario.csv
- elimina USER_49_CB (se elimina tambien en el paper original)


In [ ]:

from pathlib import Path
import re
import numpy as np
import pandas as pd

BASE = Path(r"C:\Users\oieru\OneDrive\Escritorio\MASTER\00-TFM\00-TFM_FINAL\DataBase\Limpio2")

CONV_DIR   = BASE / "Conversations"
EEG_CSV    = BASE / "EEG_Limpio_definitibo.csv"
LABELS_CSV = BASE / "phq_label_por_usuario.csv"
OUT        = BASE / "Prepared"
OUT.mkdir(parents=True, exist_ok=True)

TEXT_TAG   = "xlm-roberta-base"
SPEECH_TAG = "wav2vec2-large-xlsr-53"
MAX_DIFF_SECONDS = 120
USER_TO_REMOVE = "USER_49_CB"

print("Carpeta de salida:", OUT)

Carpeta de salida: C:\Users\oieru\OneDrive\Escritorio\MASTER\00-TFM\00-TFM_FINAL\DataBase\Limpio2\Prepared



La única normalización que se hace es sobre el identificador de usuario.  
Los avatares se mantienen tal como aparecen en `GuideOfTimes.csv` y `EEG_Limpio_definitibo.csv`.

In [ ]:

# FUNCIONES AUXILIARES

def normalize_user(x):
    """Ejemplo: User_01_CB_Conversations -> USER_01_CB."""
    return str(x).strip().upper().replace("_CONVERSATIONS", "")


def read_guide(path):
    """Lee GuideOfTimes.csv y conserva el avatar sin renombrarlo."""
    g = pd.read_csv(path, sep=";")
    g["RealTimeStr"] = pd.to_datetime(g["RealTimeStr"], errors="coerce")
    g = g.dropna(subset=["RealTimeStr", "Event"]).copy()
    g["avatar"] = g["Event"].astype(str).str.strip()
    return g[["RealTimeStr", "avatar"]]


def datetime_from_folder(name):
    """Extrae fecha/hora de carpetas tipo User_01_CB_20221116_094233."""
    m = re.search(r"(\d{8}_\d{6})", str(name))
    if m is None:
        return pd.NaT
    return pd.to_datetime(m.group(1), format="%Y%m%d_%H%M%S", errors="coerce")


def nearest_avatar(dt, guide):
    """Devuelve el avatar cuya hora en GuideOfTimes está más cerca de la carpeta."""
    if pd.isna(dt) or guide.empty:
        return None, np.inf
    diffs = (guide["RealTimeStr"] - dt).abs().dt.total_seconds()
    i = diffs.idxmin()
    return guide.loc[i, "avatar"], float(diffs.loc[i])


def load_embedding(path):
    """Carga un .npy como vector 1D."""
    return np.asarray(np.load(path, allow_pickle=True), dtype=float).reshape(-1)


def collect_embeddings(tag, prefix):
    """Crea una tabla con una fila por usuario-avatar para text o speech."""
    rows = []

    for user_folder in sorted(CONV_DIR.glob("*_Conversations")):
        subject_id = normalize_user(user_folder.name)
        guide_path = user_folder / "GuideOfTimes.csv"
        if not guide_path.exists():
            continue

        guide = read_guide(guide_path)

        for conv_folder in sorted(p for p in user_folder.iterdir() if p.is_dir()):
            files = list(conv_folder.rglob(f"*{tag}*.npy"))
            if not files:
                continue

            avatar, diff = nearest_avatar(datetime_from_folder(conv_folder.name), guide)
            if diff > MAX_DIFF_SECONDS:
                continue

            emb = load_embedding(files[0])
            row = {"subject_id": subject_id, "avatar": avatar, "_diff": diff}
            row.update({f"{prefix}_{i}": v for i, v in enumerate(emb)})
            rows.append(row)

    df = pd.DataFrame(rows)
    if df.empty:
        return df

    # Si hubiera dos carpetas enlazadas al mismo usuario-avatar, se conserva la más cercana por hora.
    df = (df.sort_values("_diff")
            .drop_duplicates(["subject_id", "avatar"], keep="first")
            .drop(columns="_diff")
            .reset_index(drop=True))
    return df


def embedding_cols(df, prefix):
    """Selecciona solo columnas reales de embedding: text_0, text_1..."""
    return [c for c in df.columns if re.fullmatch(fr"{prefix}_\d+", str(c))]

 Cargar datos



In [ ]:

labels = pd.read_csv(LABELS_CSV).rename(columns={"user": "subject_id"})
labels["subject_id"] = labels["subject_id"].apply(normalize_user)
labels = labels[["subject_id", "label"]].drop_duplicates("subject_id")

eeg = pd.read_csv(EEG_CSV).rename(columns={"user": "subject_id"})
eeg["subject_id"] = eeg["subject_id"].apply(normalize_user)
eeg["avatar"] = eeg["avatar"].astype(str).str.strip()
eeg = eeg.drop(columns=[c for c in ["phq", "label"] if c in eeg.columns])
eeg = eeg.drop_duplicates(["subject_id", "avatar"], keep="first").reset_index(drop=True)

text = collect_embeddings(TEXT_TAG, "text")
speech = collect_embeddings(SPEECH_TAG, "speech")

# USER_49_CB se elimina de las tablas basadas en embeddings, pero no de EEG.
text = text[text["subject_id"] != USER_TO_REMOVE].copy()
speech = speech[speech["subject_id"] != USER_TO_REMOVE].copy()

print("Labels:", labels.shape)
print("EEG:", eeg.shape)
print("Text:", text.shape)
print("Speech:", speech.shape)

Labels: (102, 2)
EEG: (558, 29)
Text: (600, 770)
Speech: (600, 1026)


Crear y guardar las 5 tablas

Los CSV finales contienen solo:
subject_id,
avatar,
label,


In [ ]:

# CREAR Y GUARDAR TABLAS FINALES

text_cols = embedding_cols(text, "text")
speech_cols = embedding_cols(speech, "speech")

# 1) Solo text
text_only = text[["subject_id", "avatar"] + text_cols].merge(labels, on="subject_id", how="left")
text_only = text_only[["subject_id", "avatar", "label"] + text_cols]
text_only.to_csv(OUT / "01_only_text.csv", index=False)

# 2) Solo speech
speech_only = speech[["subject_id", "avatar"] + speech_cols].merge(labels, on="subject_id", how="left")
speech_only = speech_only[["subject_id", "avatar", "label"] + speech_cols]
speech_only.to_csv(OUT / "02_only_speech.csv", index=False)

# 3) Text + speech
text_speech = text_only.merge(
    speech_only.drop(columns="label"),
    on=["subject_id", "avatar"],
    how="inner"
)
text_speech.to_csv(OUT / "03_text_speech.csv", index=False)

# 4) Solo EEG. Aquí se mantiene USER_49_CB.
eeg_only = eeg.merge(labels, on="subject_id", how="left")
eeg_cols = [c for c in eeg_only.columns if c not in ["subject_id", "avatar", "label"]]
eeg_only = eeg_only[["subject_id", "avatar", "label"] + eeg_cols]
eeg_only.to_csv(OUT / "04_only_eeg.csv", index=False)

# 5) Text + speech + EEG.
# Si no hay EEG para una conversación, se conserva la fila y las columnas EEG quedan como NA.
eeg_features = eeg_only.drop(columns="label")
text_speech_eeg = text_speech.merge(
    eeg_features,
    on=["subject_id", "avatar"],
    how="left"
)
text_speech_eeg.to_csv(OUT / "05_text_speech_eeg.csv", index=False)

print("Archivos guardados en:", OUT)

Archivos guardados en: C:\Users\oieru\OneDrive\Escritorio\MASTER\00-TFM\00-TFM_FINAL\DataBase\Limpio2\Prepared


 Resumen rápido

Para ver si las tablas se han creado correctamente.

In [ ]:

tables = {
    "01_only_text": text_only,
    "02_only_speech": speech_only,
    "03_text_speech": text_speech,
    "04_only_eeg": eeg_only,
    "05_text_speech_eeg": text_speech_eeg,
}

for name, df in tables.items():
    print(
        f"{name:22s} filas={df.shape[0]:4d} | "
        f"usuarios={df['subject_id'].nunique():3d} | "
        f"label_NA={df['label'].isna().sum()}"
    )

print("\nUsuarios con nº de avatares distinto de 6, ignorando USER_10_CB2 y USER_39_CB2:")
ignore = {"USER_10_CB2", "USER_39_CB2"}
for name in ["01_only_text", "02_only_speech", "03_text_speech", "05_text_speech_eeg"]:
    counts = tables[name].groupby("subject_id")["avatar"].nunique()
    problem = counts[(counts != 6) & (~counts.index.isin(ignore))]
    print(f"{name}: {len(problem)} usuarios")
    if len(problem):
        display(problem.sort_index())

01_only_text           filas= 600 | usuarios=101 | label_NA=0
02_only_speech         filas= 600 | usuarios=101 | label_NA=0
03_text_speech         filas= 600 | usuarios=101 | label_NA=0
04_only_eeg            filas= 558 | usuarios= 94 | label_NA=0
05_text_speech_eeg     filas= 600 | usuarios=101 | label_NA=0

Usuarios con nº de avatares distinto de 6, ignorando USER_10_CB2 y USER_39_CB2:
01_only_text: 0 usuarios
02_only_speech: 0 usuarios
03_text_speech: 0 usuarios
05_text_speech_eeg: 0 usuarios
